In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# Configuration
# ============================================================
INPUT_FILE   = 'analysis_dataset.csv'
SHAPEFILE    = 'tl_2021_us_county/tl_2021_us_county.shp'
MAP1_FILE    = 'map1_radiologist_density.png'
MAP2_FILE    = 'map2_ai_mortality_reduction.png'

BETA         = -2.2383
DELTA_AI_MOD = 5.1688   # Moderate scenario (nonzero median)

# ============================================================
# Helper: reproject Alaska and Hawaii for inset display
# ============================================================
def make_lower48_ak_hi(gdf):
    """Split into lower 48, Alaska, Hawaii for standard US map layout."""
    gdf = gdf.to_crs('EPSG:4326')
    lower48 = gdf[~gdf['state'].isin(['AK', 'HI'])].copy()
    alaska   = gdf[gdf['state'] == 'AK'].copy()
    hawaii   = gdf[gdf['state'] == 'HI'].copy()

    # Reproject lower 48 to Albers Equal Area
    lower48 = lower48.to_crs('EPSG:5070')

    # Scale + shift Alaska
    alaska = alaska.to_crs('EPSG:3338')
    alaska['geometry'] = alaska['geometry'].scale(0.35, 0.35, origin=(0,0))
    alaska['geometry'] = alaska['geometry'].translate(-2800000, -2000000)
    alaska = alaska.to_crs('EPSG:5070')

    # Scale + shift Hawaii
    hawaii = hawaii.to_crs('EPSG:32604')
    hawaii['geometry'] = hawaii['geometry'].translate(5200000, 500000)
    hawaii = hawaii.to_crs('EPSG:5070')

    return lower48, alaska, hawaii

# ============================================================
# Step 1: Load and prepare data
# ============================================================
print("Loading data...")
df = pd.read_csv(INPUT_FILE, low_memory=False, dtype={'FIPS': str})
gdf_raw = gpd.read_file(SHAPEFILE)
gdf_raw = gdf_raw.rename(columns={'GEOID': 'FIPS'})

# Merge all counties (not just OLS sample) for complete map
gdf = gdf_raw.merge(df, on='FIPS', how='inner')
print(f"Counties for mapping: {len(gdf):,}")

# Compute AI mortality reduction (moderate scenario)
def mortality_reduction(density, delta_ai, beta=BETA):
    delta_log = np.log(density + delta_ai + 1) - np.log(density + 1)
    return -beta * delta_log   # positive = reduction

gdf['ai_reduction_moderate'] = mortality_reduction(
    gdf['density_radiologist'], DELTA_AI_MOD
)

# Shortage classification
gdf['shortage_cat'] = pd.cut(
    gdf['density_radiologist'],
    bins=[-0.001, 0.0, 1.2716, np.inf],
    labels=['Extreme shortage\n(density = 0)',
            'Severe shortage\n(density < 1.3)',
            'Non-shortage\n(density ≥ 1.3)']
)

# Split regions
lower48, alaska, hawaii = make_lower48_ak_hi(gdf)

print("Reprojection complete.")

# ============================================================
# Step 2: MAP 1 — Radiologist Density (Shortage Status)
# ============================================================
print("\nDrawing Map 1: Radiologist density / shortage status...")

colors_shortage = {
    'Extreme shortage\n(density = 0)':    '#d62728',
    'Severe shortage\n(density < 1.3)':   '#ff7f0e',
    'Non-shortage\n(density ≥ 1.3)':      '#2ca02c',
}

fig, ax = plt.subplots(1, 1, figsize=(14, 9))
ax.set_axis_off()

# Grey base layer: ALL counties from shapefile (includes CT and other missing)
gdf_all_lower48 = gdf_raw[~gdf_raw['STATEFP'].isin(['02','15','72','78','60','66','69'])].copy()
gdf_all_lower48 = gdf_all_lower48.to_crs('EPSG:5070')
gdf_all_lower48.plot(ax=ax, color='#cccccc', linewidth=0.1, edgecolor='white')
# Alaska
gdf_all_ak = gdf_raw[gdf_raw['STATEFP'] == '02'].copy()
gdf_all_ak = gdf_all_ak.to_crs('EPSG:3338')
gdf_all_ak['geometry'] = gdf_all_ak['geometry'].scale(0.35, 0.35, origin=(0,0))
gdf_all_ak['geometry'] = gdf_all_ak['geometry'].translate(-2800000, -2000000)
gdf_all_ak = gdf_all_ak.to_crs('EPSG:5070')
gdf_all_ak.plot(ax=ax, color='#cccccc', linewidth=0.1, edgecolor='white')
# Hawaii
gdf_all_hi = gdf_raw[gdf_raw['STATEFP'] == '15'].copy()
gdf_all_hi = gdf_all_hi.to_crs('EPSG:32604')
gdf_all_hi['geometry'] = gdf_all_hi['geometry'].translate(5200000, 500000)
gdf_all_hi = gdf_all_hi.to_crs('EPSG:5070')
gdf_all_hi.plot(ax=ax, color='#cccccc', linewidth=0.1, edgecolor='white')

for label, color in colors_shortage.items():
    for part in [lower48, alaska, hawaii]:
        subset = part[part['shortage_cat'] == label]
        if len(subset) > 0:
            subset.plot(ax=ax, color=color, linewidth=0.1, edgecolor='white')

# Draw state borders (lower48 only)
state_borders = lower48.dissolve(by='state')
state_borders.boundary.plot(ax=ax, color='grey', linewidth=0.4)

# Stats annotation — top right
n_extreme = (gdf['density_radiologist'] == 0).sum()
n_severe  = (gdf['density_radiologist'] < 1.2716).sum()
ax.text(0.98, 0.98,
        f"Extreme shortage (density = 0): {n_extreme:,} ({n_extreme/len(gdf)*100:.1f}%)\n"
        f"Severe shortage (density < 1.3): {n_severe:,} ({n_severe/len(gdf)*100:.1f}%)",
        transform=ax.transAxes, fontsize=10, ha='right', va='top',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='white', alpha=0.9))

# Legend — bottom left, no overlap
patches = [mpatches.Patch(color=c, label=l.replace('\n', ' '))
           for l, c in colors_shortage.items()]
patches.append(mpatches.Patch(color='#cccccc', label='Data not available'))
ax.legend(handles=patches, loc='lower left', fontsize=10,
          title='Radiologist Shortage Status', title_fontsize=11,
          framealpha=0.9, bbox_to_anchor=(0.01, 0.01))

ax.set_title('County-Level Diagnostic Radiologist Shortage Status, United States',
             fontsize=14, fontweight='bold', pad=12)

plt.tight_layout()
plt.savefig(MAP1_FILE, dpi=150, bbox_inches='tight', facecolor='white')
print(f"Map 1 saved: {MAP1_FILE}")
plt.close()

# ============================================================
# Step 3: MAP 2 — AI Mortality Reduction (Moderate Scenario)
# ============================================================
print("\nDrawing Map 2: AI mortality reduction (moderate scenario)...")

# Only counties with mortality data
gdf_mort = gdf[gdf['mortality_rate'].notna()].copy()
lower48_m, alaska_m, hawaii_m = make_lower48_ak_hi(gdf_mort)

# Grey out counties without mortality data
gdf_no_mort = gdf[gdf['mortality_rate'].isna()].copy()
lower48_nm, alaska_nm, hawaii_nm = make_lower48_ak_hi(gdf_no_mort)

fig, ax = plt.subplots(1, 1, figsize=(14, 9))
ax.set_axis_off()

# Layer 1: dark grey base for ALL counties from shapefile
# (covers CT + counties not in analysis dataset at all)
# Lower 48
gdf_all_lower48 = gdf_raw[~gdf_raw['STATEFP'].isin(['02','15','72','78','60','66','69'])].copy()
gdf_all_lower48 = gdf_all_lower48.to_crs('EPSG:5070')
gdf_all_lower48.plot(ax=ax, color='#888888', linewidth=0.1, edgecolor='white')
# Alaska (same transform as make_lower48_ak_hi)
gdf_all_ak = gdf_raw[gdf_raw['STATEFP'] == '02'].copy()
gdf_all_ak = gdf_all_ak.to_crs('EPSG:3338')
gdf_all_ak['geometry'] = gdf_all_ak['geometry'].scale(0.35, 0.35, origin=(0,0))
gdf_all_ak['geometry'] = gdf_all_ak['geometry'].translate(-2800000, -2000000)
gdf_all_ak = gdf_all_ak.to_crs('EPSG:5070')
gdf_all_ak.plot(ax=ax, color='#888888', linewidth=0.1, edgecolor='white')
# Hawaii (same transform as make_lower48_ak_hi)
gdf_all_hi = gdf_raw[gdf_raw['STATEFP'] == '15'].copy()
gdf_all_hi = gdf_all_hi.to_crs('EPSG:32604')
gdf_all_hi['geometry'] = gdf_all_hi['geometry'].translate(5200000, 500000)
gdf_all_hi = gdf_all_hi.to_crs('EPSG:5070')
gdf_all_hi.plot(ax=ax, color='#888888', linewidth=0.1, edgecolor='white')

# Layer 2: light grey for suppressed mortality counties (in analysis dataset but mortality suppressed)
for part in [lower48_nm, alaska_nm, hawaii_nm]:
    if len(part) > 0:
        part.plot(ax=ax, color='#cccccc', linewidth=0.1, edgecolor='white')

# Layer 3: choropleth for counties with mortality data
cmap = LinearSegmentedColormap.from_list(
    'reduction', ['#ffffcc', '#fd8d3c', '#800026'], N=256
)

vmin = gdf_mort['ai_reduction_moderate'].quantile(0.02)
vmax = gdf_mort['ai_reduction_moderate'].quantile(0.98)

for part in [lower48_m, alaska_m, hawaii_m]:
    if len(part) > 0:
        part.plot(ax=ax, column='ai_reduction_moderate',
                  cmap=cmap, vmin=vmin, vmax=vmax,
                  linewidth=0.1, edgecolor='white')

# State borders
state_borders_m = lower48_m.dissolve(by='state')
state_borders_m.boundary.plot(ax=ax, color='grey', linewidth=0.4)

# Colorbar
sm = plt.cm.ScalarMappable(cmap=cmap,
     norm=plt.Normalize(vmin=vmin, vmax=vmax))
sm._A = []
cbar = fig.colorbar(sm, ax=ax, fraction=0.025, pad=0.02, shrink=0.6)
cbar.set_label('Reduction in Cancer Mortality Rate\n(per 100,000 population)',
               fontsize=11)

# Legend with both grey types
patch_suppressed = mpatches.Patch(color='#cccccc', label='Mortality data suppressed (small population)')
patch_na = mpatches.Patch(color='#888888', label='Data not available (CT geographic mismatch or missing)')
ax.legend(handles=[patch_suppressed, patch_na],
          loc='lower left', fontsize=9, framealpha=0.9,
          bbox_to_anchor=(0.01, 0.01))

ax.set_title(
    f'Projected Cancer Mortality Reduction Under AI-Enabled Capacity Expansion\n'
    f'Moderate Scenario (ΔAI = 5.2 radiologists per 100,000)',
    fontsize=13, fontweight='bold', pad=12
)

plt.tight_layout()
plt.savefig(MAP2_FILE, dpi=150, bbox_inches='tight', facecolor='white')
print(f"Map 2 saved: {MAP2_FILE}")
plt.close()

print("\nDONE. Both maps saved.")

Loading data...
Counties for mapping: 3,135
Reprojection complete.

Drawing Map 1: Radiologist density / shortage status...
Map 1 saved: map1_radiologist_density.png

Drawing Map 2: AI mortality reduction (moderate scenario)...
Map 2 saved: map2_ai_mortality_reduction.png

DONE. Both maps saved.
